Search is the first stage that can answer the wrong question while every service reports healthy. This week develops a lexical retrieval baseline, explains its scoring, and introduces an evaluation method that remains useful after embeddings and agents arrive.

The implementation reference is the [Week 3 release](https://github.com/jamwithai/production-agentic-rag-course/tree/week3.0), particularly `src/services/opensearch/index_config.py`, `query_builder.py`, `client.py`, and `src/routers/search.py`. The Week 7 folder no longer contains that original index configuration, so running the old notebook against the final source would obscure the progression. The learning goal is to predict a ranking, explain a failure, and measure the effect of a change.

## Information needs and relevance judgments

A user asking about “retrieval in language models” probably wants papers describing a mechanism, not every document containing “language.” A lexical system sees query tokens and indexed terms; relevance depends on how those terms stand for the user's need.

A small evaluation set makes ranking changes comparable against fixed information needs. Exact technical phrases and acronyms test lexical precision; paraphrases expose vocabulary mismatches; questions requiring two concepts test evidence coverage; and unanswerable requests reveal how the system handles absent evidence. Each query is paired with papers or passages known to support it, so relevance can be assessed independently of the ranker's scores. A separate held-out set checks whether an improvement extends beyond the examples that guided tuning.

**Relevance labels** are a measurement instrument. A paper can be topically related without containing the evidence needed for an answer. The labels might mean “about the topic,” “contains supporting evidence,” or “sufficient to answer,” each defining a different target. Mixing those meanings makes apparent improvements difficult to interpret.

The [BEIR benchmark](https://arxiv.org/abs/2104.08663) evaluated retrieval across varied datasets and found BM25 a robust baseline. That supports starting with a lexical comparator; it does not establish that BM25 wins on this arXiv corpus. The corpus, queries, judgments, and latency constraints still determine the result.

## Analysis determines what can match

An inverted index maps terms to documents and associated statistics. Before indexing, an analyzer turns text into terms. Lowercasing can make `Transformer` match `transformer`; stemming can connect some inflections; stop-word removal can reduce common terms. Each choice also removes distinctions.

The Week 3 mapping uses analyzed `text` fields and `keyword` representations for exact values. Categories and identifiers should generally be filtered as exact values. A scientific identifier such as `2501.00001v2` should not accidentally become several unrelated tokens. Titles and abstracts, in contrast, need term-based matching.

OpenSearch's analysis output reveals the representation that scoring actually receives. If a term never reaches the index, no amount of field boosting can retrieve it. A mismatch between source and query analysis can likewise explain a missed result without implicating BM25's parameters. This illustrates a broader debugging principle: an algorithm's behavior is constrained by the representation supplied at its boundary.

## The incentives in BM25

One common BM25 formulation scores a document $d$ for query terms $q$ as

$$
\operatorname{BM25}(q,d)=\sum_{t\in q}\operatorname{IDF}(t)
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1\left(1-b+b\frac{|d|}{\overline{\ell}}\right)}.
$$

Here $f(t,d)$ counts a term in a document, $|d|$ is document length, and $\overline{\ell}$ is average length. The positive-IDF variant used in the experiment is

$$
\operatorname{IDF}(t)=\log\left(1+\frac{N-n_t+0.5}{n_t+0.5}\right),
$$

where $N$ is the document count and $n_t$ is the number containing term $t$. Rare terms carry more evidence than ubiquitous ones. Term frequency saturates: the tenth repetition adds less than the second. Length normalization reduces the advantage a long document gains by having more chances to contain query terms. Setting $b=0$ removes that normalization; increasing $k_1$ slows saturation.

OpenSearch uses BM25 by default, but exact scores also depend on field analysis, field statistics, query construction, and distributed execution. The following implementation isolates the formula with a simple tokenizer. It is not a reimplementation of every OpenSearch scoring detail.

**Experiment: compare exact wording with repetition.** 

1. Predict whether repeating “retrieval” will beat a short document containing both query terms.
2. Run the cell and compare the scores.
3. Inspect the separate term contributions to explain the result.


In [ ]:
import math
import re
from collections import Counter

documents = {
    "A": "retrieval augmented generation uses retrieved evidence",
    "B": "retrieval " * 12 + "database indexing storage systems",
    "C": "language models answer questions from external passages",
    "D": "generation uses language models",
}
tokens = {key: re.findall(r"[a-z]+", text.lower()) for key, text in documents.items()}
counts = {key: Counter(words) for key, words in tokens.items()}
average_length = sum(map(len, tokens.values())) / len(tokens)

def bm25(query, key, k1=1.2, b=0.75):
    score = 0.0
    for term in set(re.findall(r"[a-z]+", query.lower())):
        df = sum(term in words for words in counts.values())
        idf = math.log(1 + (len(tokens) - df + 0.5) / (df + 0.5))
        tf = counts[key][term]
        norm = k1 * (1 - b + b * len(tokens[key]) / average_length)
        score += idf * tf * (k1 + 1) / (tf + norm)
    return score

ranking = sorted(documents, key=lambda key: (-bm25("retrieval generation", key), key))
for key in ranking:
    print(key, f"{bm25('retrieval generation', key):.4f}", documents[key][:65])
assert bm25("retrieval generation", "C") == 0
assert bm25("retrieval generation", "A") > bm25("retrieval generation", "B")


Reference output from isolated Python execution:

```text
A 1.5604 retrieval augmented generation uses retrieved evidence
B 1.3028 retrieval retrieval retrieval retrieval retrieval retrieval retri
D 0.8782 generation uses language models
C 0.0000 language models answer questions from external passages
```

Document C could be semantically useful, but it shares neither query term and receives zero. Document A receives evidence for two terms, while B's repetition gives diminishing returns. These are consequences of the representation and formula. A lexical miss on a paraphrase is a reason to investigate semantic retrieval; it is not a reason to discard lexical retrieval for identifiers and exact terminology.

## Field boosts encode assumptions

The release's paper-level query boosts title, abstract, and author fields with relative factors of three, two, and one. This expresses a prior that a match in the title is often more useful than one elsewhere. It does not mean a title is literally three times as relevant, or that the final score decomposes into a simple weighted count.

The `multi_match` mode and operator in [`query_builder.py`](https://github.com/jamwithai/production-agentic-rag-course/blob/week3.0/src/services/opensearch/query_builder.py) determine how evidence from several fields contributes to a match. With a permissive operator, matching any query term may admit many weak candidates; requiring every term can miss relevant paraphrases. Boosts, fuzziness, and filtering interact with those choices. Their usefulness is clearer across distinct failure categories than on a single attractive query.

Sorting by publication date answers “what is newest among matching results?” Sorting by score answers “what does this ranker consider most relevant?” Those are different products. Filters constrain eligibility and should be applied before deciding which eligible items are best. A relevance score is not a calibrated probability of correctness.

## Order and coverage measure different properties

For a set of relevant items $R$ and the first $k$ results $L_k$, precision is $|L_k\cap R|/k$ and recall is $|L_k\cap R|/|R|$. These definitions assume relevance judgments are sufficiently complete. Unjudged items are not automatically irrelevant; document the labeling convention when using a small set.

**Reciprocal rank** is the inverse of the first relevant result's rank, or zero if none appears. It emphasizes early success. It cannot tell whether all evidence needed for a multi-part question was found. Graded metrics such as nDCG can distinguish degrees of relevance, but adding a metric is useful only when the labels support its interpretation.

**Experiment: two rankers can have identical coverage and different usefulness.** Both lists below retrieve both known relevant papers by rank four. A user or context-budget-limited generator may still prefer one.


In [ ]:
relevant = {"A", "C"}
rankers = {"early": ["A", "B", "C", "D"], "late": ["B", "D", "A", "C"]}

def metrics(ranked, relevant, k):
    top = ranked[:k]
    assert len(top) == len(set(top)), "Deduplicate retrieval units before evaluation"
    hits = len(set(top) & relevant)
    first = next((rank for rank, item in enumerate(top, 1) if item in relevant), None)
    return {"precision": hits / k, "recall": hits / len(relevant),
            "reciprocal_rank": 1 / first if first else 0.0}

for name, ranked in rankers.items():
    print(name, "at 2:", metrics(ranked, relevant, 2), "at 4:", metrics(ranked, relevant, 4))
assert metrics(rankers["early"], relevant, 4)["recall"] == 1
assert metrics(rankers["late"], relevant, 4)["reciprocal_rank"] == 1 / 3


Reference output from isolated Python execution:

```text
early at 2: {'precision': 0.5, 'recall': 0.5, 'reciprocal_rank': 1.0} at 4: {'precision': 0.5, 'recall': 1.0, 'reciprocal_rank': 1.0}
late at 2: {'precision': 0.0, 'recall': 0.0, 'reciprocal_rank': 0.0} at 4: {'precision': 0.5, 'recall': 1.0, 'reciprocal_rank': 0.3333333333333333}
```

The exercise separates finding an item eventually from presenting useful evidence early. Per-query and per-category results make regressions visible: an improvement on broad questions can otherwise mask worse retrieval for exact paper IDs. Several chunks from one paper also should not inflate paper-level recall. The counting unit—paper or passage—determines what the metric measures.

## Index failures and empty results

The Week 3 indexer writes papers using the arXiv ID as the OpenSearch document ID. That helps repeated indexing update the same logical object. It does not solve the earlier database/index transaction gap or guarantee that updates become visible immediately to search.

Refresh governs when indexed changes become searchable. A write acknowledgment and a query returning the new document are distinct observations. For a tiny lab, a refresh can make the experiment deterministic; refreshing every item in a large ingestion job can reduce indexing throughput. The relevant tradeoff is indexing throughput against the application's freshness requirement.

An empty result can mean no lexical match, an empty index, a wrong index name, a failed request hidden by error handling, or a filter excluding all documents. The response contract and logs provide the evidence needed to distinguish these causes. Operational failure and absence of evidence should remain distinguishable.

## Docker lab: establish and challenge a lexical baseline

1. Use a separate `week3.0` checkout and project name `rag-week3`; stop the earlier stack first. Start the complete Compose definition, run the ingestion DAG, and verify that real records reach the paper index. Database rows and task success alone are insufficient.

    ```bash
    docker compose --project-name rag-week3 up --build -d
    docker compose --project-name rag-week3 exec airflow \
      airflow dags trigger arxiv_paper_ingestion
    curl --fail-with-body 'http://localhost:9200/_cat/indices?v'
    curl --fail-with-body http://localhost:9200/arxiv-papers/_count
    curl --fail-with-body http://localhost:9200/arxiv-papers/_mapping
    curl --fail-with-body -H 'Content-Type: application/json' \
      http://localhost:9200/arxiv-papers/_analyze \
      -d '{"analyzer":"text_analyzer","text":"Transformers retrieve retrieved retrieval"}'
    curl --fail-with-body -H 'Content-Type: application/json' \
      http://localhost:8000/api/v1/search/ \
      -d '{"query":"retrieval generation","size":5,"latest_papers":false}'
    ```

2. Use a query whose wording you have verified in one ingested paper. If an endpoint differs, inspect `http://localhost:8000/openapi.json` and the pinned router rather than borrowing a later release's URL. Retrieve one result's OpenSearch `_id` and submit a corresponding `_explain` request to inspect its score. Compare title and abstract matches with your prediction.

3. Build a fixed query-and-judgment set, then run it with default boosts and one altered boost setting. Change one assumption at a time. Retain ranks, IDs, scores, filters, latency, and index version. Randomize query order and separate warm-up runs from measured repetitions; a single request is neither a latency distribution nor evidence of general retrieval quality.

4. For failure injection, stop OpenSearch while leaving the API running and repeat the request. Record the status and body. Restore OpenSearch and confirm the same known paper is retrievable. The desired behavior is an explicit dependency failure followed by recovery, not a successful-looking empty result. If the implementation hides the failure, document it as a defect and propose a response-contract change.

## Engineering review: choose a baseline you can defend

1. Diagnose these failures separately: a relevant paper uses “external memory” while the query says “retrieval”; a paper matches “learning” hundreds of times but says little about the requested mechanism; and the right paper disappears only when a category filter is added. Each points to a different intervention—representation, scoring, or eligibility.

2. Submit a short comparison containing per-query results, a failure taxonomy, and one proposed change. A convincing argument explains why the change should help, which queries could regress, and which held-out cases will decide. Week 4 adds vectors and chunking to this baseline; keeping these measurements makes the additional complexity accountable.

## Sources and scope

- [Week 3 source](https://github.com/jamwithai/production-agentic-rag-course/tree/week3.0): mapping, query builder, stable paper IDs, and search API.
- [OpenSearch 2.19 keyword search](https://docs.opensearch.org/2.19/search-plugins/keyword-search/): BM25 as the default similarity model.
- [Thakur et al., BEIR](https://arxiv.org/abs/2104.08663): evidence for evaluating retrieval across heterogeneous tasks and retaining lexical baselines.

The formula experiment uses a small deterministic corpus and a deliberately simple analyzer. Its scores are pedagogical calculations, not expected numerical outputs from the deployed OpenSearch index.
